# Feature Corrections: Proof-of-Concept

This notebook demonstrates that post-hoc analytical corrections can substantially
reduce the rotation sensitivity (RSI) of cp_measure features.

**What this is:**
A proof-of-concept showing that corrections work mathematically, using known `angle_deg`
(available only in simulation).

**What this is NOT:**
A production-ready solution.  Production corrections belong in:
- **cp_measure**: pre-orient the image before measurement (the cleanest solution)
- **pycytominer**: post-hoc transformations applied to the feature table

See `feature_math.ipynb` for the mathematical derivations.

## Corrections implemented here

| Feature group | Correction method |
|---|---|
| `CentralMoment_p_q`, `NormalizedMoment_p_q` | Binomial moment rotation by `-angle_deg` |
| `SpatialMoment_p_q` | Translate to centroid, then apply moment rotation |
| `InertiaTensor` (all elements) | Rotate $(\mu_{02}, \mu_{11}, \mu_{20})$ then re-derive |
| `RadialDistribution_ZernikePhase_n_m` | `phase + m * angle_deg_rad` |
| `Location_*_X/Y` | Convert to distance from geometric centroid |

**Not corrected here** (no analytical post-hoc fix): Granularity, ZernikeMagnitude, Intensity edge features, HuMoments and Zernike odd-$m$ (not real rotation problems).


In [1]:
from math import comb
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATASET = Path("rotation_dataset.ome.parquet")
assert DATASET.exists(), "Run simulate_rotation_sweep.py first"

META_COLS = [
    "cell_id",
    "angle_deg",
    "aspect_ratio",
    "y_radius",
    "stain_type",
    "stain_corr",
    "seed",
]
IMAGE_COLS = ["image", "mask"]

schema = pq.read_schema(DATASET)
feature_cols = [f.name for f in schema if f.name not in META_COLS + IMAGE_COLS]

# Load full feature data (no images)
df = pq.read_table(DATASET, columns=META_COLS + feature_cols).to_pandas()
print(f"Loaded {len(df):,} rows x {len(feature_cols)} feature columns")

Loaded 115,200 rows x 372 feature columns


## Baseline RSI (before corrections)

In [2]:
def compute_rsi(data: pd.DataFrame, feats: list) -> pd.Series:
    """Compute RSI for each feature in feats."""
    grp = data.groupby("cell_id")[feats]
    within_var = grp.var().mean()
    at_zero = data[data["angle_deg"] == 0.0][feats]
    between_var = at_zero.var()
    return within_var / between_var.replace(0, np.nan)


rsi_before = compute_rsi(df, feature_cols)

# Collect the features we will try to correct
central_feats = [f for f in feature_cols if f.startswith("CentralMoment_")]
norm_feats = [f for f in feature_cols if f.startswith("NormalizedMoment_")]
spatial_feats = [f for f in feature_cols if f.startswith("SpatialMoment_")]
inertia_feats = [
    f for f in feature_cols if f.startswith("InertiaTensor_") and "Eigenvalues" not in f
]
phase_feats = [f for f in feature_cols if "ZernikePhase" in f]
loc_feats = [
    f for f in feature_cols if f.startswith("Location_") and f.endswith(("_X", "_Y"))
]

targets = (
    central_feats + norm_feats + spatial_feats + inertia_feats + phase_feats + loc_feats
)
print(f"Target features to correct: {len(targets)}")
print(f"  CentralMoment:    {len(central_feats)}")
print(f"  NormalizedMoment: {len(norm_feats)}")
print(f"  SpatialMoment:    {len(spatial_feats)}")
print(f"  InertiaTensor:    {len(inertia_feats)}")
print(f"  ZernikePhase:     {len(phase_feats)}")
print(f"  Location X/Y:     {len(loc_feats)}")
print()
print("Baseline RSI for target features:")
print(rsi_before[targets].describe().round(4).to_string())

Target features to correct: 75
  CentralMoment:    12
  NormalizedMoment: 13
  SpatialMoment:    12
  InertiaTensor:    4
  ZernikePhase:     30
  Location X/Y:     4

Baseline RSI for target features:
count    7.400000e+01
mean     3.901709e+04
std      1.503952e+05
min      0.000000e+00
25%      6.426000e-01
50%      1.169200e+00
75%      5.017200e+00
max      1.036114e+06


## Correction 1 — Central and Normalized Moments

Apply the binomial rotation formula with $\alpha = -\text{angle\_deg}$ to undo the rotation.

All $p, q \in \{0,1,2,3\}$ moment values are available in the dataset, so the formula
can be applied vectorised row-by-row.


In [3]:
def build_moment_corrector(max_p=3, max_q=3):
    """Pre-compute the correction coefficients for vectorised row application.

    Returns a list of (p, q, a, b, coeff_factor) tuples where:
        corrected_mu[(p,q)] += coeff * original_mu[(a+b, p+q-a-b)]
    (correction = rotate by -alpha, so pass cos(-alpha) = cos, sin(-alpha) = -sin)
    """
    terms = []
    for p in range(max_p + 1):
        for q in range(max_q + 1):
            for a in range(p + 1):
                for b in range(q + 1):
                    src_p = a + b
                    src_q = p + q - a - b
                    if src_p > max_p or src_q > max_q:
                        continue
                    terms.append((p, q, a, b, src_p, src_q))
    return terms


MOMENT_TERMS = build_moment_corrector()


def correct_moments_vectorised(
    df_sub: pd.DataFrame, prefix: str, alpha_col: str
) -> pd.DataFrame:
    """Correct all moment features with given prefix using rotation by -alpha_col.

    Returns a copy of df_sub with corrected moment columns.
    """
    # Parse available (p,q) from column names
    available = {}
    for col in df_sub.columns:
        if col.startswith(prefix + "_"):
            parts = col.split("_")
            if len(parts) >= 3:
                try:
                    p, q = int(parts[-2]), int(parts[-1])
                    available[(p, q)] = col
                except ValueError:
                    pass

    alpha = np.deg2rad(-df_sub[alpha_col].values)  # negative = undo rotation
    c = np.cos(alpha)
    s = np.sin(alpha)  # sin(-alpha) = -sin(alpha) handled via formula

    result = df_sub.copy()

    # Accumulate corrected values
    corrected = {k: np.zeros(len(df_sub)) for k in available}
    for p, q, a, b, src_p, src_q in MOMENT_TERMS:
        if (p, q) not in available or (src_p, src_q) not in available:
            continue
        src_col = available[(src_p, src_q)]
        # coeff for rotation by (-alpha): cos(-alpha)=c, sin(-alpha)=-s
        coeff = (
            comb(p, a)
            * comb(q, b)
            * ((-1) ** b)
            * (c ** (a + q - b))
            * ((-s) ** (p - a + b))
        )
        corrected[(p, q)] += coeff * df_sub[src_col].values

    for (p, q), vals in corrected.items():
        result[available[(p, q)]] = vals.astype(np.float32)

    return result


print("Applying moment corrections (may take ~30s)...")
df_corr = correct_moments_vectorised(df, "CentralMoment", "angle_deg")
df_corr = correct_moments_vectorised(df_corr, "NormalizedMoment", "angle_deg")
print("Done.")

Applying moment corrections (may take ~30s)...


Done.


In [4]:
# RSI after central + normalized moment corrections
rsi_after_moments = compute_rsi(df_corr, central_feats + norm_feats)

comp = pd.DataFrame(
    {
        "before": rsi_before[central_feats + norm_feats],
        "after": rsi_after_moments,
    }
).round(4)
comp["reduction_pct"] = ((1 - comp["after"] / comp["before"]) * 100).round(1)
print("Central + Normalized Moment corrections (RSI before vs. after):")
print(comp.sort_values("before", ascending=False).to_string())

Central + Normalized Moment corrections (RSI before vs. after):
                            before          after  reduction_pct
CentralMoment_1_1     1.036114e+06      54.338402     100.000000
CentralMoment_1_3     4.216638e+05  231421.281250      45.099998
NormalizedMoment_3_3  3.468833e+05   34822.234375      90.000000
NormalizedMoment_3_1  1.528853e+05    2042.127563      98.699997
NormalizedMoment_1_3  4.337257e+04   31260.722656      27.900000
NormalizedMoment_1_1  3.035392e+04       2.231900     100.000000
NormalizedMoment_2_2  2.542386e+02     159.097305      37.400002
CentralMoment_2_1     3.482130e+01       7.763600      77.699997
CentralMoment_2_3     1.811100e+01       2.592700      85.699997
NormalizedMoment_3_2  1.515460e+01       3.799700      74.900002
CentralMoment_1_0     1.368830e+01       4.076600      70.199997
NormalizedMoment_3_0  8.229300e+00       0.731300      91.099998
NormalizedMoment_2_0  7.877600e+00       0.000200     100.000000
CentralMoment_2_0     6.39

## Correction 2 — Spatial Moments

**Step 1:** Convert $M_{pq} \to \mu_{pq}$ using the centroid.
**Step 2:** Apply moment rotation correction.

Note: Only the **corrected** central moments are stored back; we don't rewrite spatial moments
(they encode position, which is inherently frame-dependent).


In [5]:
def spatial_to_central(df_sub: pd.DataFrame) -> pd.DataFrame:
    """Compute central moments from spatial moments for order <= 3."""
    res = df_sub.copy()
    M = {}  # noqa: N806
    for col in df_sub.columns:
        if col.startswith("SpatialMoment_"):
            parts = col.split("_")
            if len(parts) == 3:
                p, q = int(parts[1]), int(parts[2])
                M[(p, q)] = df_sub[col].values.astype(float)

    M00 = M.get((0, 0), np.ones(len(df_sub)))  # noqa: N806
    xbar = M.get((1, 0), np.zeros(len(df_sub))) / np.where(M00 != 0, M00, 1)
    ybar = M.get((0, 1), np.zeros(len(df_sub))) / np.where(M00 != 0, M00, 1)

    max_order = 3
    for p in range(max_order + 1):
        for q in range(max_order + 1):
            if p + q < 2 or (p, q) not in M:
                continue
            mu_pq = np.zeros(len(df_sub))
            for a in range(p + 1):
                for b in range(q + 1):
                    if (a, b) in M:
                        mu_pq += (
                            comb(p, a)
                            * comb(q, b)
                            * ((-xbar) ** (p - a))
                            * ((-ybar) ** (q - b))
                            * M[(a, b)]
                        )
            res[f"SpatialMoment_{p}_{q}_central"] = mu_pq.astype(np.float32)

    return res


# Convert spatial -> central, then rotate
df_sp = spatial_to_central(df)
central_from_spatial = [c for c in df_sp.columns if c.endswith("_central")]

# Apply rotation correction to the derived central moments
alpha = np.deg2rad(-df_sp["angle_deg"].values)
c, s = np.cos(alpha), np.sin(alpha)


# Build a small moment dict structure for each pair
def rotate_col_set(df_sub, available_cols, alpha_arr):
    """available_cols: dict {(p,q): col_name}"""
    c_ = np.cos(alpha_arr)
    s_ = -np.sin(alpha_arr)  # rotating by -alpha
    result = {k: np.zeros(len(df_sub)) for k in available_cols}
    for p, q, a, b, sp, sq in MOMENT_TERMS:
        if (p, q) not in available_cols or (sp, sq) not in available_cols:
            continue
        coeff = (
            comb(p, a)
            * comb(q, b)
            * ((-1) ** b)
            * (c_ ** (a + q - b))
            * (s_ ** (p - a + b))
        )
        result[(p, q)] += coeff * df_sub[available_cols[(sp, sq)]].values
    return result


sp_central_available = {}
for col in central_from_spatial:
    parts = col.replace("_central", "").split("_")
    if len(parts) == 3:
        p, q = int(parts[1]), int(parts[2])
        sp_central_available[(p, q)] = col

corrected_sp = rotate_col_set(
    df_sp, sp_central_available, np.deg2rad(-df_sp["angle_deg"].values)
)

# Store corrected values back
for (p, q), vals in corrected_sp.items():
    col = sp_central_available[(p, q)]
    df_corr[col] = vals.astype(np.float32)

# Evaluate: compare RSI of raw SpatialMoment vs the corrected central version
rsi_sp_raw = compute_rsi(df, spatial_feats)
rsi_sp_corr = compute_rsi(df_corr, list(sp_central_available.values()))
print("Spatial Moment RSI (raw spatial vs corrected central):")
for (p, q), col in sorted(sp_central_available.items()):
    raw_col = f"SpatialMoment_{p}_{q}"
    if raw_col in rsi_sp_raw.index:
        print(
            f"  {raw_col}: {rsi_sp_raw[raw_col]:.4f}  ->"  # noqa: E501
            f"  {col}: {rsi_sp_corr.get(col, float('nan')):.4f}"
        )

Spatial Moment RSI (raw spatial vs corrected central):
  SpatialMoment_0_2: 0.1424  ->  SpatialMoment_0_2_central: 0.0000
  SpatialMoment_0_3: 0.1699  ->  SpatialMoment_0_3_central: 5.3006
  SpatialMoment_1_1: 0.2775  ->  SpatialMoment_1_1_central: 54.3385
  SpatialMoment_1_2: 0.4216  ->  SpatialMoment_1_2_central: 0.1026
  SpatialMoment_1_3: 0.4960  ->  SpatialMoment_1_3_central: 231663.1250
  SpatialMoment_2_0: 6.6718  ->  SpatialMoment_2_0_central: 0.0000
  SpatialMoment_2_1: 2.8875  ->  SpatialMoment_2_1_central: 7.7610
  SpatialMoment_2_2: 2.8621  ->  SpatialMoment_2_2_central: 2.5484
  SpatialMoment_2_3: 2.8607  ->  SpatialMoment_2_3_central: 2.5990


## Correction 3 — Inertia Tensor

The inertia tensor elements are directly related to 2nd-order central moments:
- $I_{00} = \mu_{02}$, $I_{11} = \mu_{20}$, $I_{01} = I_{10} = -\mu_{11}$

After correcting `CentralMoment_p_q`, we can re-derive the corrected tensor elements.


In [6]:
# Re-derive corrected InertiaTensor from corrected CentralMoments
df_corr["InertiaTensor_0_0"] = df_corr["CentralMoment_0_2"].values.astype(np.float32)
df_corr["InertiaTensor_1_1"] = df_corr["CentralMoment_2_0"].values.astype(np.float32)
df_corr["InertiaTensor_0_1"] = (-df_corr["CentralMoment_1_1"]).values.astype(np.float32)
df_corr["InertiaTensor_1_0"] = df_corr["InertiaTensor_0_1"].values

rsi_inertia_before = rsi_before[inertia_feats]
rsi_inertia_after = compute_rsi(df_corr, inertia_feats)

print("InertiaTensor RSI before vs. after:")
comp_i = pd.DataFrame({"before": rsi_inertia_before, "after": rsi_inertia_after}).round(
    4
)
comp_i["reduction_pct"] = ((1 - comp_i["after"] / comp_i["before"]) * 100).round(1)
print(comp_i.to_string())

InertiaTensor RSI before vs. after:
                        before      after  reduction_pct
InertiaTensor_0_0       0.1857   0.000000          100.0
InertiaTensor_0_1  427775.0000  54.338402          100.0
InertiaTensor_1_0  427775.0000  54.338402          100.0
InertiaTensor_1_1       8.6025   0.000000          100.0


## Correction 4 — RadialDistribution ZernikePhase

Under rotation by $\alpha$, the Zernike coefficient $c_{nm} \to e^{-im\alpha} c_{nm}$.
With the cp_measure convention `phase = arctan2(Re, Im)`, the phase shifts as:
$$\phi'_{nm} = \phi_{nm} + m\alpha$$

So the correction is: $\phi^{\text{corr}}_{nm} = \phi_{nm} - m\alpha = \phi_{nm} + m \cdot (-\text{angle\_deg\_rad})$.

We verify the sign empirically below before applying.


In [7]:
# Empirical sign verification for ZernikePhase
# Pick one cell, one (n,m) pair with m != 0
test_cell = 1
test_feat = "RadialDistribution_ZernikePhase_2_2"
m_order = 2

sub = df[df["cell_id"] == test_cell].sort_values("angle_deg")
alpha_rad = np.deg2rad(sub["angle_deg"].values)
phase_measured = sub[test_feat].values
phase_0 = phase_measured[0]

# Predict under both sign conventions
pred_minus = phase_0 - m_order * alpha_rad  # phase decreases
pred_plus = phase_0 + m_order * alpha_rad  # phase increases


# Wrap to [-pi, pi]
def wrap(x):
    return (x + np.pi) % (2 * np.pi) - np.pi


residual_minus = np.abs(wrap(phase_measured - wrap(pred_minus))).mean()
residual_plus = np.abs(wrap(phase_measured - wrap(pred_plus))).mean()

print(f"ZernikePhase_{m_order} sign check (cell {test_cell}):")
print(f"  Mean |residual| with sign = -m*alpha: {residual_minus:.5f} rad")
print(f"  Mean |residual| with sign = +m*alpha: {residual_plus:.5f} rad")
sign_str = "+" if residual_plus < residual_minus else "-"
print(f"  => Better fit with sign = {sign_str}m*alpha")
sign = 1 if residual_plus < residual_minus else -1

ZernikePhase_2 sign check (cell 1):
  Mean |residual| with sign = -m*alpha: 1.31066 rad
  Mean |residual| with sign = +m*alpha: 0.88361 rad
  => Better fit with sign = +m*alpha


In [8]:
# Apply ZernikePhase correction to all phase features
alpha_rad_col = np.deg2rad(df["angle_deg"].values)

for feat in phase_feats:
    parts = feat.replace("RadialDistribution_ZernikePhase_", "").split("_")
    if len(parts) < 2:
        continue
    n_val, m_val = int(parts[0]), int(parts[1])
    if m_val == 0:
        continue  # phase undefined for m=0; skip

    corrected = df[feat].values + sign * m_val * alpha_rad_col
    # Wrap to [-pi, pi]
    corrected = (corrected + np.pi) % (2 * np.pi) - np.pi
    df_corr[feat] = corrected.astype(np.float32)

rsi_phase_before = rsi_before[phase_feats]
rsi_phase_after = compute_rsi(df_corr, phase_feats)

comp_phase = pd.DataFrame(
    {
        "m": [int(f.split("_")[-1]) for f in phase_feats],
        "before": rsi_phase_before.values,
        "after": rsi_phase_after.values,
    },
    index=phase_feats,
).round(4)
comp_phase["reduction_pct"] = (
    (1 - comp_phase["after"] / comp_phase["before"]) * 100
).round(1)
print("ZernikePhase RSI before vs. after:")
print(comp_phase.sort_values("before", ascending=False).to_string())

ZernikePhase RSI before vs. after:
                                     m  before   after  reduction_pct
RadialDistribution_ZernikePhase_4_4  4  4.0341  0.6565      83.699997
RadialDistribution_ZernikePhase_6_6  6  2.4747  0.3224      87.000000
RadialDistribution_ZernikePhase_2_2  2  2.1727  0.2486      88.599998
RadialDistribution_ZernikePhase_4_2  2  1.9288  0.3689      80.900002
RadialDistribution_ZernikePhase_8_8  8  1.8245  0.4740      74.000000
RadialDistribution_ZernikePhase_5_5  5  1.4086  0.7323      48.000000
RadialDistribution_ZernikePhase_6_4  4  1.2421  0.2496      79.900002
RadialDistribution_ZernikePhase_9_9  9  1.2337  0.7063      42.700001
RadialDistribution_ZernikePhase_6_2  2  1.1799  0.2178      81.500000
RadialDistribution_ZernikePhase_1_1  1  1.1726  0.5346      54.400002
RadialDistribution_ZernikePhase_8_4  4  1.1657  0.2431      79.099998
RadialDistribution_ZernikePhase_5_3  3  1.1601  0.6359      45.200001
RadialDistribution_ZernikePhase_8_6  6  1.1366  0.2606 

## Correction 5 — Location Features

Convert `Location_*_X` and `Location_*_Y` to distance from the geometric centroid.
The geometric centroid is $(\bar x, \bar y) = (M_{10}/M_{00},\; M_{01}/M_{00})$.


In [9]:
# Compute centroid from spatial moments
centroid_x = (df["SpatialMoment_1_0"] / df["SpatialMoment_0_0"]).values
centroid_y = (df["SpatialMoment_0_1"] / df["SpatialMoment_0_0"]).values

new_loc_feats = []
for base in ["CenterMassIntensity", "MaxIntensity"]:
    xcol = f"Location_{base}_X"
    ycol = f"Location_{base}_Y"
    if xcol not in df.columns or ycol not in df.columns:
        continue
    dist_col = f"Location_{base}_Distance"
    dx = df[xcol].values - centroid_x
    dy = df[ycol].values - centroid_y
    df_corr[dist_col] = np.sqrt(dx**2 + dy**2).astype(np.float32)
    new_loc_feats.append(dist_col)
    print(f"Added {dist_col}")

rsi_loc_before = rsi_before[loc_feats]
rsi_loc_after = compute_rsi(df_corr, new_loc_feats)

print()
print("Location RSI (raw X/Y vs corrected Distance):")
for xcol, ycol, dist_col in zip(
    [f for f in loc_feats if f.endswith("_X")],
    [f for f in loc_feats if f.endswith("_Y")],
    new_loc_feats,
):
    print(f"  {xcol}: {rsi_before[xcol]:.3f}  {ycol}: {rsi_before[ycol]:.3f}")
    print(f"  -> {dist_col}: {rsi_loc_after.get(dist_col, float('nan')):.4f}")
    print()

Added Location_CenterMassIntensity_Distance
Added Location_MaxIntensity_Distance

Location RSI (raw X/Y vs corrected Distance):
  Location_CenterMassIntensity_X: 0.588  Location_CenterMassIntensity_Y: 5.345
  -> Location_CenterMassIntensity_Distance: 0.1204

  Location_MaxIntensity_X: 0.629  Location_MaxIntensity_Y: 3.749
  -> Location_MaxIntensity_Distance: 0.6086



## Summary: before vs. after corrections

In [10]:
# Collect all corrected features and build a comparison table
all_corrected_targets = (
    central_feats + norm_feats + spatial_feats + inertia_feats + phase_feats + loc_feats
)
rsi_final = compute_rsi(df_corr, all_corrected_targets + new_loc_feats)

rows = []
for f in all_corrected_targets:
    rows.append(
        {
            "feature": f,
            "rsi_before": float(rsi_before.get(f, np.nan)),
            "rsi_after": float(rsi_final.get(f, np.nan)),
        }
    )
for f in new_loc_feats:
    orig_base = f.replace("_Distance", "")
    rows.append(
        {
            "feature": f + " (new)",
            "rsi_before": float(rsi_before.get(f.replace("Distance", "X"), np.nan)),
            "rsi_after": float(rsi_final.get(f, np.nan)),
        }
    )

summary_df = pd.DataFrame(rows)
summary_df["reduction_factor"] = (
    summary_df["rsi_before"] / summary_df["rsi_after"].replace(0, np.nan)
).round(1)

print("Correction summary (features with RSI > 0.01 before correction):")
show = summary_df[summary_df["rsi_before"] > 0.01].sort_values(
    "rsi_before", ascending=False
)
pd.set_option("display.float_format", "{:.4g}".format)
print(show.to_string(index=False))

Correction summary (features with RSI > 0.01 before correction):
                                    feature  rsi_before  rsi_after  reduction_factor
                          CentralMoment_1_1   1.036e+06      54.34         1.907e+04
                          InertiaTensor_1_0   4.278e+05      54.34              7872
                          InertiaTensor_0_1   4.278e+05      54.34              7872
                          CentralMoment_1_3   4.217e+05  2.314e+05               1.8
                       NormalizedMoment_3_3   3.469e+05  3.482e+04                10
                       NormalizedMoment_3_1   1.529e+05       2042              74.9
                       NormalizedMoment_1_3   4.337e+04  3.126e+04               1.4
                       NormalizedMoment_1_1   3.035e+04      2.232          1.36e+04
                       NormalizedMoment_2_2       254.2      159.1               1.6
                          CentralMoment_2_1       34.82      7.764               4.5


In [11]:
# Visual summary: before/after RSI distribution (log scale)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

before_vals = np.log10(summary_df["rsi_before"].dropna().clip(lower=1e-6))
after_vals = np.log10(summary_df["rsi_after"].dropna().clip(lower=1e-6))

axes[0].hist(
    before_vals, bins=40, color="coral", alpha=0.85, edgecolor="none", label="Before"
)
axes[0].hist(
    after_vals, bins=40, color="steelblue", alpha=0.65, edgecolor="none", label="After"
)
axes[0].axvline(0, color="black", linestyle="--", alpha=0.7, label="RSI = 1.0")
axes[0].axvline(
    np.log10(0.01), color="green", linestyle="--", alpha=0.7, label="RSI = 0.01"
)
axes[0].set_xlabel("log10(RSI)")
axes[0].set_ylabel("Features")
axes[0].set_title("Distribution of RSI: before vs. after corrections")
axes[0].legend()

# Per-feature scatter: before vs after
non_trivial = summary_df[
    (summary_df["rsi_before"] > 0.01) & summary_df["rsi_after"].notna()
]
axes[1].scatter(
    np.log10(non_trivial["rsi_before"].clip(lower=1e-6)),
    np.log10(non_trivial["rsi_after"].clip(lower=1e-6)),
    alpha=0.6,
    s=20,
    color="purple",
)
lims = (-3, 7)
axes[1].plot(lims, lims, "--", color="gray", alpha=0.5, label="no change")
axes[1].plot(
    lims, [x - 2 for x in lims], ":", color="green", alpha=0.5, label="100x improvement"
)
axes[1].set_xlim(lims)
axes[1].set_ylim(lims)
axes[1].set_xlabel("log10(RSI before)")
axes[1].set_ylabel("log10(RSI after)")
axes[1].set_title("RSI before vs. after (one point per corrected feature)")
axes[1].legend()

plt.tight_layout()
plt.savefig("/tmp/corrections_summary.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved /tmp/corrections_summary.png")

Saved /tmp/corrections_summary.png


/tmp/ipykernel_40688/8686535.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Next steps

### For cp_measure

The cleanest production implementation is to add an optional `canonical_orientation` flag
that pre-rotates the image to `Orientation = 0` before computing all features.
This makes ALL features (including Granularity, Texture, edge Intensity) rotation-invariant
with a single change, without requiring per-feature post-hoc transforms.

### For pycytominer

Alternatively, add a `correct_rotation()` transform that applies the analytical corrections
derived here to any feature table.  This is more flexible (can be applied retroactively)
but requires all required source features to be present in the table.

### Features to add to cp_measure

- `Location_CenterMassDistance`, `Location_MaxIntensityDistance` — rotation-invariant
  location descriptors (distance from geometric centroid).
